# Data Quality — Detecting Missing Values

Energy time series often have gaps caused by sensor outages or communication
failures. The `data_quality` module provides functions to **detect**, **fill**,
and **visualize** these gaps.

This example uses hourly indoor temperature data bundled with the package.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Load sample data

The bundled `flat_temp.csv` contains hourly indoor temperature readings
with real gaps from sensor outages.

In [ ]:
import pandas as pd
from importlib import resources

# Load the bundled sample data
data_path = resources.files("pyedautils") / "data" / "flat_temp.csv"
df = pd.read_csv(data_path, sep=";", index_col="timestamp", parse_dates=True)
df.columns = ["temperature"]
print(f"Rows: {len(df)}, Date range: {df.index[0]} – {df.index[-1]}")
df.head()

## Detect gap durations

`calc_gap_duration` computes the time difference between consecutive rows
and a rolling median for comparison.

In [ ]:
from pyedautils.data_quality import calc_gap_duration

gaps = calc_gap_duration(df)
gaps[gaps["gapDuration"] > 60].head()

## Histogram of gap durations

A log-scaled histogram of the rolling median gap duration reveals the
dominant sampling rate and any outlier gaps.

In [ ]:
import plotly.express as px

fig = px.histogram(
    gaps,
    x="gapDuration",
    nbins=100,
    log_y=True,
    labels={"gapDuration": "Gap duration [s]", "count": "Count"},
    title="<b>Histogram of gap durations</b>",
)
fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_title="Count (log scale)",
)
fig.show()

## Fill gaps with NaN

`fill_missing_values_with_na` inserts NaN rows at the expected timestamps
where data is missing.

In [ ]:
from pyedautils.data_quality import fill_missing_values_with_na

df_filled = fill_missing_values_with_na(df)
print(f"Before fill: {len(df)} rows, After fill: {len(df_filled)} rows")
print(f"NaN rows added: {df_filled['temperature'].isna().sum()}")

## Check NaN percentage

`calc_isna_percentage` gives a quick summary of data completeness.

In [ ]:
from pyedautils.data_quality import calc_isna_percentage

pct = calc_isna_percentage(df_filled, column="temperature")
print(f"Missing: {pct}%")

## Visualize missing values

`plot_missing_values` creates an interactive Plotly chart with red bands
highlighting the NaN regions.

In [ ]:
from pyedautils.data_quality import plot_missing_values

fig = plot_missing_values(df_filled, column="temperature", ylab="Temperature [°C]")
fig.show()